# Cattle Re-ID — Etapa 3: comparación de losses (CE / ArcFace / SupCon / Triplet)

Misma augmentation fuerte + backbone + sampler PK en las 4; se entrena en **CMPD300**, se **congela**, y se evalúa clustering en **Zenodo** (sin etiquetas). Métrica primaria: **HDBSCAN ARI** (media ± desvío sobre seeds). Baselines: ImageNet y DINOv2.

**Modelos cacheados a Drive** (celda 2): DINOv2 y ResNet-ImageNet se descargan una sola vez. Los checkpoints entrenados se guardan en `outputs/checkpoints` (symlink a Drive) → persisten.

## 1. Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/cattle_reid')   # ajustá si tu carpeta se llama distinto
assert DRIVE_ROOT.is_dir(), f'No existe {DRIVE_ROOT}.'
print('contenido:', [p.name for p in DRIVE_ROOT.iterdir()])

## 2. Cachear modelos en Drive (ANTES de importar torch)

Setea `TORCH_HOME` (ResNet ImageNet) y `HF_HOME` (DINOv2) apuntando a Drive: se descargan una vez y se reusan en cada sesión.

In [ ]:
import os
CACHE = DRIVE_ROOT / 'model_cache'
(CACHE/'torch').mkdir(parents=True, exist_ok=True)
(CACHE/'hf').mkdir(parents=True, exist_ok=True)
os.environ['TORCH_HOME'] = str(CACHE/'torch')
os.environ['HF_HOME']    = str(CACHE/'hf')
print('TORCH_HOME=', os.environ['TORCH_HOME'])
print('HF_HOME   =', os.environ['HF_HOME'])

## 3. GPU

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'Sin GPU: Entorno -> Cambiar tipo -> GPU.'
print('GPU:', torch.cuda.get_device_name(0))

## 4. Código + dependencias

In [ ]:
import shutil
os.chdir('/content')
REPO_URL = 'https://github.com/benjaminvitale/tp-final-vision2-Pujol-Vitale.git'
BRANCH   = 'main'
REPO_DIR = '/content/tp-final-vision2-Pujol-Vitale'
if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1
!pip -q install 'transformers==4.40.2'

## 5. Persistir outputs en Drive (checkpoints entrenados no se pierden)

In [ ]:
DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'
for sub in ('checkpoints', 'logs', 'results'):
    ds = DRIVE_OUTPUTS / sub; ds.mkdir(parents=True, exist_ok=True)
    ls = Path(REPO_DIR) / 'outputs' / sub
    if ls.exists() and not ls.is_symlink(): shutil.rmtree(ls)
    if not ls.exists(): os.symlink(ds, ls, target_is_directory=True)
    print(f'outputs/{sub} -> {ds}')

## 6. Datos: CMPD300 (source, entrenamiento) + Zenodo (target, eval)

In [ ]:
import zipfile
IMG_EXTS = {'.jpg','.jpeg','.png','.bmp','.webp'}

# --- CMPD300 (source) ---
CMPD_ZIP = DRIVE_ROOT / 'CMPD300_Baseline.zip'
CMPD_LOCAL = Path('/content/cmpd300')
if not CMPD_LOCAL.exists():
    CMPD_LOCAL.mkdir(parents=True); !unzip -q "{CMPD_ZIP}" -d "{CMPD_LOCAL}"
cand = [CMPD_LOCAL, CMPD_LOCAL/'Baseline'] + [p for p in CMPD_LOCAL.rglob('*') if p.is_dir()]
CMPD_DIR = next((p for p in cand if (p/'train').is_dir()), None)
assert CMPD_DIR, 'No encuentro train/ en CMPD300.'
SOURCE_TRAIN = CMPD_DIR / 'train'
print('SOURCE_TRAIN:', SOURCE_TRAIN, '| ids:', len([d for d in SOURCE_TRAIN.iterdir() if d.is_dir()]))

# --- Zenodo (target) ---
MUZZLE_ZIP = DRIVE_ROOT / 'BeefCattle_Muzzle_Individualized.zip'
MUZZLE_LOCAL = Path('/content/muzzle')
if not MUZZLE_LOCAL.exists():
    MUZZLE_LOCAL.mkdir(parents=True); !unzip -q "{MUZZLE_ZIP}" -d "{MUZZLE_LOCAL}"
def find_root(base):
    for p in [base, *[d for d in base.rglob('*') if d.is_dir()]]:
        subs = [d for d in p.iterdir() if d.is_dir()]
        if len(subs) >= 100 and any(d.name.lower().startswith('cattle') for d in subs): return p
    raise RuntimeError('No encuentro cattle_*')
TARGET_DIR = find_root(MUZZLE_LOCAL)
print('TARGET_DIR:', TARGET_DIR, '| ids:', len([d for d in TARGET_DIR.iterdir() if d.is_dir()]))

## 7. Validación rápida del pipeline (smoke, ~1 min)

Antes de quemar GPU: confirma que las 4 losses entrenan y guardan checkpoint.

In [ ]:
for L in ['ce','arcface','supcon','triplet']:
    print('==== smoke', L, '====')
    !python scripts/13_train_loss.py --train-dir "{SOURCE_TRAIN}" --loss {L} --smoke --out /tmp/smoke_{L}.pt 2>&1 | tail -2

## 8. Entrenar las 4 losses

**Empezá con 1 seed** para tener la primera tabla; después subí `SEEDS` a [0,1,2] para el media±desvío. Cada loss = un entrenamiento full de ResNet-50 sobre CMPD300 (~1–2 h en T4), así que 4 losses × N seeds es la parte cara. Los checkpoints ya hechos se saltean.

In [ ]:
SEEDS  = [0]                 # subí a [0,1,2] para varianza
LOSSES = ['ce','arcface','supcon','triplet']
EPOCHS = 80
import subprocess
for seed in SEEDS:
    for loss in LOSSES:
        out = Path('outputs/checkpoints')/f'cmpd300_{loss}_s{seed}.pt'
        if out.exists():
            print('ya existe, salteo:', out.name); continue
        print('==== entrenando', loss, 'seed', seed, '====')
        subprocess.run(['python','scripts/13_train_loss.py','--train-dir',str(SOURCE_TRAIN),
                        '--loss',loss,'--seed',str(seed),'--epochs',str(EPOCHS),
                        '--out',str(out)], check=True)

## 9. Evaluar (dedup pHash → HDBSCAN ARI media±desvío + baselines)

In [ ]:
!python scripts/14_eval_losses.py \
    --target-dir "{TARGET_DIR}" \
    --ckpt-dir outputs/checkpoints \
    --losses ce arcface supcon triplet \
    --seeds 0 \
    --baselines imagenet dinov2

## 10. Resultados

In [ ]:
import json
res = json.loads(Path('outputs/results/14_loss_comparison.json').read_text())
print('eval set:', res['n_eval_images'], 'imgs (pHash:', res['phash'], ')')
print('\nbaselines:')
for b,m in res['baselines'].items():
    print(f"  {b:12} HDBSCAN ARI={m.get('hdbscan_ari')}  kmeans={m.get('kmeans_ari')}")
ce = res['per_loss'].get('ce',{}).get('hdbscan_ari',{}).get('mean')
print('\nlosses (HDBSCAN ARI mean±std):')
for loss,s in res['per_loss'].items():
    a = s['hdbscan_ari']; d = f"  Δce={a['mean']-ce:+.3f}" if ce is not None else ''
    print(f"  {loss:12} {a['mean']}±{a['std']} (n={s['n_seeds']}){d}")

## Cómo leer

- **HDBSCAN ARI** es la vara (no Rank-1: el probe de borde mostró que el Rank-1 se infla con contexto/fondo).
- **Δce** aísla el efecto de la loss del efecto de la augmentation. Si CE ya sube fuerte y las demás casi no se diferencian → *lo que importa es forzar el foco en el hocico, no la loss* (hallazgo igual de válido).
- Una loss "gana" solo si supera a ImageNet/DINOv2 **fuera del desvío**.